# Posture statistická pipeline (paired_test)

Tento notebook:
- načte data z `paired_test`
- spustí `paired_hypotheses_pipeline.py`
- uloží výsledky do výstupní složky
- rovnou zobrazí hlavní tabulky výsledků

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

project_root = Path.cwd()
paired_dir = project_root / "paired_test"
input_csv = paired_dir / "Final_data_frame_filtered_enrollment_pdg_merged_fes_merged_P-N_duration.csv"
pipeline_script = paired_dir / "paired_hypotheses_pipeline.py"
out_dir = paired_dir / "paired_delta_outputs_notebook"

print(f"Project root: {project_root}")
print(f"Input CSV exists: {input_csv.exists()} -> {input_csv}")
print(f"Pipeline exists: {pipeline_script.exists()} -> {pipeline_script}")
print(f"Output dir: {out_dir}")

In [ ]:
cmd = [
    sys.executable,
    str(pipeline_script),
    "--csv", str(input_csv),
    "--out", str(out_dir),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print("Exit code:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
if result.stderr.strip():
    print("\nSTDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Pipeline skončila chybou. Viz výstup výše.")

In [ ]:
run_report_path = out_dir / "run_report.json"
results_dir = out_dir / "results_tables"
figures_dir = out_dir / "figures"

run_report = json.loads(run_report_path.read_text(encoding="utf-8"))
print("RUN REPORT:")
print(json.dumps(run_report, indent=2, ensure_ascii=False))

print("\nPočet CSV výsledků:", len(list(results_dir.glob("*.csv"))))
print("Počet obrázků:", len(list(figures_dir.glob("*.png"))))

sorted([p.name for p in results_dir.glob("*.csv")])

In [ ]:
def show_csv(name: str, n: int = 20):
    path = results_dir / name
    if not path.exists():
        print(f"{name}: soubor neexistuje")
        return None
    df = pd.read_csv(path)
    print(f"\n===== {name} | shape={df.shape} =====")
    display(df.head(n))
    return df

primary = show_csv("PRIMARY_delta_tests_with_FDR.csv")
h1_terms = show_csv("H1_terms.csv")
h2 = show_csv("H2_gait_links.csv")
h3_primary = show_csv("H3_primary_moca.csv")
h4_omnibus = show_csv("H4_group_omnibus_on_deltas.csv")
h6 = show_csv("H6_LEDD_associations.csv")
h8_meta = show_csv("H8_cluster_meta.csv")